# Surya OCR Testing

**WeatherSpeak PH** — Gemma 4 Hackathon

## About Surya OCR

[Surya](https://github.com/VikParuchuri/surya) is a modern AI-native OCR toolkit that:
- Supports **90+ languages**
- **Beats Google Cloud Vision** on benchmarks
- Includes layout analysis, table recognition, reading order detection
- Purpose-built for **document understanding** (not scene text)
- 19.6k stars, actively maintained

### Why Test Surya?
Surya represents the current **state-of-the-art in open-source document OCR**. If it performs well on PAGASA bulletins, it's a strong production candidate.

## 1. Install Surya OCR

In [ ]:
!pip install -q surya-ocr torch torchvision

In [ ]:
import os
import json
from pathlib import Path
from PIL import Image
import time

from surya.foundation import FoundationPredictor
from surya.recognition import RecognitionPredictor
from surya.detection import DetectionPredictor

print("✓ Surya OCR imported successfully")

## 2. Load Sample Data

Load the sample images prepared in the setup notebook.

In [ ]:
# Load metadata
data_dir = Path("../data")
metadata_path = data_dir / "sample_metadata.json"

with open(metadata_path, 'r') as f:
    metadata = json.load(f)

samples = metadata['samples']
print(f"Loaded {len(samples)} sample images")

# Create output directory for results
output_dir = data_dir / "surya_results"
output_dir.mkdir(exist_ok=True)

## 3. Initialize Surya Models

Surya uses separate models for detection and recognition.

In [ ]:
%%time
# Initialize predictors
print("Loading Surya models (this may take a moment on first run)...")

foundation_predictor = FoundationPredictor()
recognition_predictor = RecognitionPredictor(foundation_predictor)
detection_predictor = DetectionPredictor()

print("✓ Models loaded")

## 4. Run OCR on Sample Images

Process each sample image and extract text.

In [ ]:
results = []

for i, sample in enumerate(samples, 1):
    print(f"\n{'='*60}")
    print(f"Processing {i}/{len(samples)}: {sample['filename']}")
    print(f"{'='*60}")
    
    # Load image
    image = Image.open(sample['image_path'])
    
    # Run OCR
    start_time = time.time()
    predictions = recognition_predictor([image], det_predictor=detection_predictor)
    processing_time = time.time() - start_time
    
    # Extract text from predictions
    prediction = predictions[0]
    text_lines = prediction['text_lines']
    full_text = '\n'.join([line['text'] for line in text_lines])
    
    # Store results
    result = {
        'filename': sample['filename'],
        'storm': sample['storm'],
        'processing_time': processing_time,
        'num_lines': len(text_lines),
        'full_text': full_text,
        'text_lines': text_lines
    }
    results.append(result)
    
    # Display summary
    print(f"\n📊 Results:")
    print(f"  Processing time: {processing_time:.2f}s")
    print(f"  Lines detected: {len(text_lines)}")
    print(f"  Characters extracted: {len(full_text)}")
    print(f"\n📄 Extracted text (first 500 characters):")
    print("-" * 60)
    print(full_text[:500])
    print("-" * 60)

## 5. Save Results

Save OCR results for comparison in the analysis notebook.

In [ ]:
# Save results to JSON
results_path = output_dir / "surya_ocr_results.json"
with open(results_path, 'w', encoding='utf-8') as f:
    json.dump(results, f, indent=2, ensure_ascii=False)

print(f"✓ Results saved to: {results_path}")

# Also save full text files for easy reading
for result in results:
    text_path = output_dir / f"{Path(result['filename']).stem}_text.txt"
    with open(text_path, 'w', encoding='utf-8') as f:
        f.write(result['full_text'])

print(f"✓ Text files saved to: {output_dir}")

## 6. Performance Summary

Aggregate performance metrics across all samples.

In [ ]:
import statistics

processing_times = [r['processing_time'] for r in results]
line_counts = [r['num_lines'] for r in results]

print("\n" + "="*60)
print("SURYA OCR PERFORMANCE SUMMARY")
print("="*60)
print(f"\n📊 Processing Time:")
print(f"  Average: {statistics.mean(processing_times):.2f}s")
print(f"  Min: {min(processing_times):.2f}s")
print(f"  Max: {max(processing_times):.2f}s")
print(f"\n📄 Content:")
print(f"  Average lines per page: {statistics.mean(line_counts):.0f}")
print(f"  Total samples processed: {len(results)}")
print(f"\n✓ All results saved to: {output_dir.absolute()}")

## 7. Visual Inspection

Display one sample with its extracted text for manual quality assessment.

In [ ]:
# Display first sample
if results:
    sample_result = results[0]
    sample_info = samples[0]
    
    print(f"Sample: {sample_result['filename']}")
    print(f"Storm: {sample_result['storm']}")
    print(f"\n" + "="*60)
    
    # Show image
    img = Image.open(sample_info['image_path'])
    display(img.resize((600, int(600 * img.size[1] / img.size[0]))))
    
    # Show extracted text
    print("\n📄 EXTRACTED TEXT:")
    print("="*60)
    print(sample_result['full_text'])
    print("="*60)

## 8. Preliminary Assessment

### ✅ Strengths
- Modern AI-native approach
- Good multilingual support
- Active development and maintenance
- Built-in layout analysis

### ❓ Questions for Comparison
- How does accuracy compare to PaddleOCR?
- Can Gemma 4 Vision match this accuracy?
- Is the processing speed acceptable for production?
- How well does it handle tables and coordinates?

### 📝 Next Steps
Proceed to **Notebook 03** to test PaddleOCR on the same samples for comparison.